# Chatbot para consultar manuales técnicos

Este notebook crea un asistente RAG (*Retrieval-Augmented Generation*): busca información en un conjunto de datos público de Hugging Face y entrega los fragmentos encontrados a Ollama para redactar una respuesta fundamentada. También puedes cambiar la fuente a tus propios PDF.

**Flujo:** cargar manuales → extraer texto → dividir en fragmentos → crear embeddings e indexarlos → recuperar fragmentos pertinentes → responder mostrando las fuentes.

> **Ollama y Colab:** Google Colab ejecuta el notebook en una máquina remota. Ollama instalado en tu computador no queda disponible automáticamente allí. La sección siguiente instala Ollama dentro del runtime de Colab. Si trabajas localmente, omite esa instalación y configura `OLLAMA_URL` con la dirección de tu servicio.

## 1. Preparar el entorno

Ejecuta las celdas en orden. Descargar los modelos puede tardar y consumir varios GB. El índice de búsqueda se implementa con Python, sin ChromaDB; así se evita el error de DLL bloqueada de `rpds` que puede aparecer en Windows.

In [1]:
%pip -q install pymupdf requests


Note: you may need to restart the kernel to use updated packages.


In [2]:
import math
import os
import platform
import shutil
import subprocess
import time
from pathlib import Path

import fitz
import requests

OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434").rstrip("/")
CHAT_MODEL = "llama3.2:3b"
EMBED_MODEL = "nomic-embed-text"

if shutil.which("ollama") is None:
    if platform.system() == "Windows":
        raise RuntimeError("No se encontró Ollama. Instálalo para Windows y vuelve a ejecutar esta celda.")
    subprocess.run(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"], check=True)

ollama_process = None
try:
    requests.get(f"{OLLAMA_URL}/api/tags", timeout=2).raise_for_status()
except requests.RequestException:
    ollama_process = subprocess.Popen(
        ["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT
    )
    for _ in range(60):
        if ollama_process.poll() is not None:
            raise RuntimeError("Ollama terminó antes de iniciar.")
        try:
            requests.get(f"{OLLAMA_URL}/api/tags", timeout=2).raise_for_status()
            break
        except requests.RequestException:
            time.sleep(2)
    else:
        raise TimeoutError("Ollama no respondió en el tiempo esperado.")

for model in (CHAT_MODEL, EMBED_MODEL):
    subprocess.run(["ollama", "pull", model], check=True)

print(f"Ollama listo: {OLLAMA_URL}; modelo: {CHAT_MODEL}; embeddings: {EMBED_MODEL}")


Ollama listo: http://127.0.0.1:11434; modelo: llama3.2:3b; embeddings: nomic-embed-text


## 2. Elegir la fuente de información

Por defecto se descargan registros del dataset público [`Shekswess/technical-manuals`](https://huggingface.co/datasets/Shekswess/technical-manuals), alojado en Hugging Face. **Aviso:** su ficha lo describe como un conjunto sintético de 1.000 textos en inglés sobre documentación técnica e informática; no son manuales oficiales de máquinas industriales. Es útil para probar el flujo RAG, no para instrucciones de operación o seguridad. Para trabajar con una máquina real, usa sus manuales oficiales y cambia `DATA_SOURCE` a `"pdf"`.

In [3]:
DATA_SOURCE = "huggingface"  # opciones: "huggingface" o "pdf"
MAX_DATASET_ROWS = 200  # limita las descargas y el tiempo para este ejercicio
DATASET_ID = "Shekswess/technical-manuals"

if DATA_SOURCE == "huggingface":
    if MAX_DATASET_ROWS < 1:
        raise ValueError("MAX_DATASET_ROWS debe ser al menos 1.")
    pages = []
    api_url = "https://datasets-server.huggingface.co/rows"
    for offset in range(0, MAX_DATASET_ROWS, 100):
        length = min(100, MAX_DATASET_ROWS - offset)
        response = requests.get(
            api_url,
            params={"dataset": DATASET_ID, "config": "default", "split": "train",
                    "offset": offset, "length": length},
            timeout=60,
        )
        response.raise_for_status()
        rows = response.json().get("rows", [])
        if not rows:
            break
        for item in rows:
            row = item["row"]
            text = row.get("text", "").strip()
            if text:
                text = f"Tema: {row.get('topic', '')}. Palabras clave: {row.get('keyword', '')}.\n{text}"
                pages.append({"source": DATASET_ID, "location": f"registro {item['row_idx'] + 1}", "text": text})
    print(f"Registros cargados desde Hugging Face: {len(pages)}")
elif DATA_SOURCE == "pdf":
    try:
        from google.colab import files
        uploaded = files.upload()
        pdf_paths = [name for name in uploaded if name.lower().endswith(".pdf")]
    except ImportError:
        pdf_paths = [str(path) for path in Path(".").glob("*.pdf")]
    if not pdf_paths:
        raise ValueError("No hay PDF. En Colab, súbelos en el diálogo; localmente, colócalos junto al notebook.")
    pages = []
    for pdf_path in pdf_paths:
        with fitz.open(pdf_path) as pdf:
            for page_number, page in enumerate(pdf, start=1):
                text = page.get_text("text").strip()
                if text:
                    pages.append({"source": Path(pdf_path).name, "location": f"página {page_number}", "text": text})
else:
    raise ValueError("DATA_SOURCE debe ser 'huggingface' o 'pdf'.")

if not pages:
    raise ValueError("La fuente no devolvió texto para indexar.")
print("Muestra de texto:", pages[0]["text"][:500])


Registros cargados desde Hugging Face: 200
Muestra de texto: Tema: Technical Manuals. Palabras clave: multilingual documentation management.
Effective multilingual documentation management is crucial for technical manuals that need to be distributed across different regions and languages. It involves creating, translating, and maintaining documentation in multiple languages to cater to a global audience. This process ensures that the technical information is accurate, consistent, and accessible to users worldwide.


## 3. Dividir e indexar el contenido

Se conserva la página de cada fragmento para mostrar referencias en las respuestas. El índice se mantiene en memoria durante esta sesión de Colab; después de reiniciar, carga e indexa de nuevo los archivos.

In [4]:
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200

def split_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError("Se requiere size > overlap >= 0.")
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            boundary = text.rfind(" ", start, end)
            if boundary > start:
                end = boundary
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks

documents, metadatas = [], []
for page in pages:
    for chunk_number, chunk in enumerate(split_text(page["text"]), start=1):
        documents.append(chunk)
        metadatas.append({"source": page["source"], "location": page["location"], "chunk": chunk_number})

if not documents:
    raise ValueError("No se generaron fragmentos a partir de los manuales.")

def create_embeddings(texts):
    response = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={"model": EMBED_MODEL, "input": texts},
        timeout=300,
    )
    response.raise_for_status()
    embeddings = response.json().get("embeddings")
    if not embeddings or len(embeddings) != len(texts):
        raise RuntimeError("Ollama no devolvió un embedding por fragmento.")
    return embeddings

embedding_vectors = []
BATCH_SIZE = 32
for start in range(0, len(documents), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(documents))
    embedding_vectors.extend(create_embeddings(documents[start:end]))

if len(embedding_vectors) != len(documents):
    raise RuntimeError("La cantidad de embeddings no coincide con la cantidad de fragmentos.")
print(f"Ãndice listo: {len(embedding_vectors)} fragmentos")


Ãndice listo: 200 fragmentos


## 4. Consultar el chatbot

El asistente responde en español con el contexto recuperado e indica el origen y la ubicación (página o registro). Si la fuente no contiene la respuesta, debe decirlo en vez de inventar información.

In [ ]:
TOP_K = 4

def ask_manuals(question):
    question = question.strip()
    if not question:
        raise ValueError("Escribe una pregunta antes de consultar.")
    if not embedding_vectors:
        raise RuntimeError("El índice está vacío. Ejecuta primero la carga e indexación de documentos.")

    query_embedding = create_embeddings([question])[0]
    def cosine_similarity(vector_a, vector_b):
        dot = sum(a * b for a, b in zip(vector_a, vector_b))
        norm_a = math.sqrt(sum(value * value for value in vector_a))
        norm_b = math.sqrt(sum(value * value for value in vector_b))
        return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

    ranked = sorted(
        ((cosine_similarity(query_embedding, vector), index)
         for index, vector in enumerate(embedding_vectors)),
        reverse=True,
    )[:min(TOP_K, len(embedding_vectors))]
    retrieved = [(documents[index], metadatas[index]) for _, index in ranked]
    context_parts = []
    for index, (text, metadata) in enumerate(retrieved, start=1):
        context_parts.append(
            f"[Fuente {index}: {metadata['source']}, {metadata['location']}]\n{text}"
        )
    context = "\n\n".join(context_parts)
    prompt = f"""Eres un asistente técnico que responde en español usando exclusivamente el contexto proporcionado.
Si el contexto no contiene la respuesta, dilo claramente y no inventes instrucciones ni valores.
Cita las afirmaciones técnicas usando [1], [2], etc. Si hay riesgo de seguridad, indica que se debe seguir el procedimiento oficial del fabricante y las normas de la planta.

CONTEXTO:
{context}

PREGUNTA: {question}

RESPUESTA (en español, concisa y con citas):"""

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={"model": CHAT_MODEL, "prompt": prompt, "stream": False,
              "options": {"temperature": 0.1}},
        timeout=300,
    )
    response.raise_for_status()
    answer = response.json().get("response", "").strip()
    if not answer:
        raise RuntimeError("Ollama devolvió una respuesta vacía.")

    print("Respuesta:\n", answer)
    print("\nFuentes recuperadas:")
    for index, (_, metadata) in enumerate(retrieved, start=1):
        print(f"[{index}] {metadata['source']} — {metadata['location']}")
    return answer

ask_manuals("¿Qué puedes hacer?")


Respuesta:
 Un chatbot es un programa informático diseñado para simular conversaciones con usuarios, a menudo a través de interfaces de voz o texto. Los chatbots pueden ser entrenados para responder a preguntas frecuentes, proporcionar información y realizar tareas específicas. Según [1], un chatbot puede ser clasificado en diferentes tipos, como chatbots de texto, chatbots de voz y chatbots de mensajería.

Los chatbots pueden ser utilizados en diversas aplicaciones, como la atención al cliente, la venta en línea y la generación de contenido. Sin embargo, es importante destacar que los chatbots no son inteligencia artificial (IA) en sí mismos, sino que son herramientas que pueden ser entrenadas para simular la inteligencia artificial.

Es importante mencionar que, aunque los chatbots pueden ser útiles, no deben ser considerados como una sustitución de la inteligencia humana. En algunos casos, es recomendable seguir el procedimiento oficial del fabricante y las normas de la planta para 

'Un chatbot es un programa informático diseñado para simular conversaciones con usuarios, a menudo a través de interfaces de voz o texto. Los chatbots pueden ser entrenados para responder a preguntas frecuentes, proporcionar información y realizar tareas específicas. Según [1], un chatbot puede ser clasificado en diferentes tipos, como chatbots de texto, chatbots de voz y chatbots de mensajería.\n\nLos chatbots pueden ser utilizados en diversas aplicaciones, como la atención al cliente, la venta en línea y la generación de contenido. Sin embargo, es importante destacar que los chatbots no son inteligencia artificial (IA) en sí mismos, sino que son herramientas que pueden ser entrenadas para simular la inteligencia artificial.\n\nEs importante mencionar que, aunque los chatbots pueden ser útiles, no deben ser considerados como una sustitución de la inteligencia humana. En algunos casos, es recomendable seguir el procedimiento oficial del fabricante y las normas de la planta para garanti

## 5. Evaluar y mejorar

Prueba preguntas concretas sobre los registros descargados y comprueba que cada cita corresponda al origen y registro indicados. Recuerda que el dataset de ejemplo es sintético, está en inglés y no contiene procedimientos validados de máquinas. Para uso real, cambia `DATA_SOURCE` a `"pdf"` y carga manuales oficiales. Para una versión más completa puedes añadir OCR para PDF escaneados, persistencia de los embeddings y preguntas de prueba con respuestas verificadas.

> Este prototipo sirve como apoyo de consulta, no sustituye los procedimientos de seguridad ni la capacitación para intervenir maquinaria.